In [1]:
!pip install -q "starlette<1" "fastapi<1" --upgrade
!pip install -q gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.6.0 which is incompatible.
datasets 2.2.1 requires huggingface-hub<1.0.0,>=0.1.0, but you have huggingface-hub 1.28.0 which is incompatible.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 1.28.0 which is incompatible.


In [2]:
# Standard Library Imports
import boto3
from datetime import datetime
import json
import os
import signal
import subprocess

# Gradio
import gradio as gr

**Insight — the app auto-discovers the team's live endpoint(s) via `list_endpoints` filtered by the `team09` tag, rather than requiring a hard-coded endpoint name.** This makes the client resilient to the endpoint being redeployed under a new name (e.g., after the Notebook 03 vs Notebook 05 pipeline runs), as long as the tag is preserved.

In [3]:
# Configuration
REGION = "ap-southeast-1"

# -------------------------------------------------------------------
# TODO: Students should change these values for their own team.
# Example:
# TEAM_ID = "team01"
# STUDENT_ID = "s101"
# ENDPOINT_NAME = "iti113-team01-heart-disease"
# -------------------------------------------------------------------
TEAM_ID = "team09"
STUDENT_ID = "s901"
ENDPOINT_NAME = f"iti113-{TEAM_ID}-bank-fraud-detection"

RELAY_API_KEY = "3I8TPcnY6K9b2BUeqU7epJJvvcN_ZE1j6LSNk8DgTX6VnvR1"

print("Region:", REGION)
print("Team ID:", TEAM_ID)
print("Student ID:", STUDENT_ID)

sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)
sts = boto3.client("sts", region_name=REGION)

print("AWS identity:", sts.get_caller_identity()["Arn"])

response = sm.list_endpoints(
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=100
)

team_endpoints = []

for ep in response["Endpoints"]:
    endpoint_name = ep["EndpointName"]

    # Only show endpoints that follow the team naming convention
    if TEAM_ID in endpoint_name.lower():
        team_endpoints.append(ep)

print(f"\nEndpoints found for {TEAM_ID}: {len(team_endpoints)}\n")

for ep in team_endpoints:
    print("Endpoint name:", ep["EndpointName"])
    print("Status:", ep["EndpointStatus"])
    print("Creation time:", ep["CreationTime"])
    print("Last modified:", ep["LastModifiedTime"])
    print("-" * 80)


Region: ap-southeast-1
Team ID: team09
Student ID: s901
AWS identity: arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team09/SageMaker



Endpoints found for team09: 1

Endpoint name: iti113-team09-bank-fraud-detection
Status: InService
Creation time: 2026-08-22 17:53:07.029000+00:00
Last modified: 2026-08-22 17:55:50.558000+00:00
--------------------------------------------------------------------------------


In [5]:
try:
    for ep in team_endpoints:
        ENDPOINT_NAME = ep["EndpointName"]
        endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        print("\nEndpoint name:", endpoint_desc["EndpointName"])
        print("Status:", endpoint_desc["EndpointStatus"])
        print("Creation time:", endpoint_desc["CreationTime"])
        print("Last modified:", endpoint_desc["LastModifiedTime"])
except Exception as e:
    print("Unable to describe endpoint.")
    print("Check that ENDPOINT_NAME is correct and that your role has permission to access it.")
    print("Error:", e)


Endpoint name: iti113-team09-bank-fraud-detection
Status: InService
Creation time: 2026-08-22 17:53:07.029000+00:00
Last modified: 2026-08-22 17:55:50.558000+00:00


**Insight — `describe_endpoint` is used to confirm the discovered endpoint is `InService` and inspect its configuration before any prediction traffic is sent**, avoiding confusing errors from calling an endpoint that is still creating, updating, or has failed.

In [6]:
for ep in team_endpoints:
    ENDPOINT_NAME = ep["EndpointName"]

    print("=" * 100)
    print("Endpoint name:", ENDPOINT_NAME)

    try:
        # 1. Describe endpoint
        endpoint_desc = sm.describe_endpoint(
            EndpointName=ENDPOINT_NAME
        )

        endpoint_config_name = endpoint_desc["EndpointConfigName"]

        print("Endpoint status:", endpoint_desc["EndpointStatus"])
        print("Endpoint config:", endpoint_config_name)

        # 2. Describe endpoint config
        endpoint_config = sm.describe_endpoint_config(
            EndpointConfigName=endpoint_config_name
        )

        production_variants = endpoint_config.get("ProductionVariants", [])

        if not production_variants:
            print("No production variants found.")
            continue

        # 3. Loop through each production variant
        for variant in production_variants:
            print("-" * 80)

            variant_name = variant.get("VariantName")
            model_name = variant.get("ModelName")

            print("Variant name:", variant_name)
            print("Model name:", model_name)

            # Check whether endpoint is serverless or real-time instance
            if "ServerlessConfig" in variant:
                serverless_config = variant["ServerlessConfig"]
                print("Endpoint type: Serverless")
                print("Memory size:", serverless_config.get("MemorySizeInMB"), "MB")
                print("Max concurrency:", serverless_config.get("MaxConcurrency"))
            else:
                print("Endpoint type: Real-time instance")
                print("Instance type:", variant.get("InstanceType"))
                print("Initial instance count:", variant.get("InitialInstanceCount"))

            if not model_name:
                print("No model name found for this variant.")
                continue

            # 4. Describe model
            model_desc = sm.describe_model(
                ModelName=model_name
            )

            # 5. Handle PrimaryContainer or Containers
            if "PrimaryContainer" in model_desc:
                container = model_desc["PrimaryContainer"]

                print("Container format: PrimaryContainer")
                print("Primary container image:", container.get("Image", "Not shown"))
                print("Model artifact S3 URI:", container.get("ModelDataUrl", "Not shown"))

            elif "Containers" in model_desc:
                for i, container in enumerate(containers, start=1):
                    print(f"Container {i} image:", container.get("Image", "Not shown"))
                    print(f"Container {i} model artifact S3 URI:", container.get("ModelDataUrl", "Not shown"))
                
                    if "ModelPackageName" in container:
                        model_package_arn = container["ModelPackageName"]
                        print(f"Container {i} model package ARN:", model_package_arn)
                
                        try:
                            package_desc = sm.describe_model_package(
                                ModelPackageName=model_package_arn
                            )
                
                            inference_spec = package_desc.get("InferenceSpecification", {})
                            package_containers = inference_spec.get("Containers", [])
                
                            print("Model package approval status:", package_desc.get("ModelApprovalStatus", "Not shown"))
                
                            for j, pkg_container in enumerate(package_containers, start=1):
                                print(f"Package container {j} image:", pkg_container.get("Image", "Not shown"))
                                print(f"Package container {j} model artifact S3 URI:", pkg_container.get("ModelDataUrl", "Not shown"))
                
                        except Exception as e:
                            print("Unable to inspect model package details.")
                            print("Error:", e)
    except Exception as e:
        print("Unable to inspect this endpoint's model details.")
        print("Error:", e)

print("=" * 100)
print("Endpoint inspection completed.")

Endpoint name: iti113-team09-bank-fraud-detection
Endpoint status: InService
Endpoint config: iti113-team09-bank-fraud-detection
--------------------------------------------------------------------------------
Variant name: AllTraffic
Model name: team09-BankFraudDetection-2026-08-22-17-53-05-558
Endpoint type: Serverless
Memory size: 2048 MB
Max concurrency: 5


Unable to inspect this endpoint's model details.
Error: name 'containers' is not defined
Endpoint inspection completed.


In [7]:
# Test the live endpoint
rt = boto3.client("sagemaker-runtime", region_name=REGION)

# High-risk profile: failed attempts, high frequency transaction, pin changed
high_risk = {
    "transaction_id": "TXN9990000001",
    "customer_id": "CUST99121959",
    "transaction_date": "2024-08-17",
    "transaction_time": "02:53:00",
    "hour_of_day": 2,
    "is_weekend": 1,
    "is_night_transaction": 1,
    "country": "Brazil",
    "city": "Rio",
    "merchant_category": "Crypto Exchange",
    "payment_method": "Crypto",
    "device_type": "Mobile",
    "customer_age": 25,
    "credit_score": 600,
    "account_age_years": 1.0,
    "account_balance": 500.0,
    "transaction_amount": 5000.0,
    "num_prev_transactions": 10,
    "transaction_freq_monthly": 2,
    "distance_from_home_km": 100.0,
    "time_since_last_txn_hrs": 24.0,
    "is_international": 1,
    "failed_attempts": 3,
    "pin_changed_recently": 1,
}
resp = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(high_risk),
)
result = json.loads(resp["Body"].read())[0]
print("HIGH-RISK PROFILE (night, international, failed attempts, pin changed, crypto)")
print(f'  Prediction  : {result["label"]}')
print(f'  Probability : {result["probability"]:.1%}')

HIGH-RISK PROFILE (night, international, failed attempts, pin changed, crypto)
  Prediction  : Fraud
  Probability : 95.4%


In [8]:
# Test the live endpoint
rt = boto3.client("sagemaker-runtime", region_name=REGION)

# Low-risk profile: local transaction for grocery
low_risk = {
    "transaction_id": "TXN9990000001",
    "customer_id": "CUST99121959",
    "transaction_date": "2024-08-17",
    "transaction_time": "13:53:00",
    "hour_of_day": 13,
    "is_weekend": 0,
    "is_night_transaction": 1,
    "country": "Brazil",
    "city": "Rio",
    "merchant_category": "Grocery",
    "payment_method": "Debit Card",
    "device_type": "Mobile",
    "customer_age": 25,
    "credit_score": 600,
    "account_age_years": 1.0,
    "account_balance": 500.0,
    "transaction_amount": 50.0,
    "num_prev_transactions": 180,
    "transaction_freq_monthly": 2,
    "distance_from_home_km": 1.0,
    "time_since_last_txn_hrs": 240.0,
    "is_international": 0,
    "failed_attempts": 1,
    "pin_changed_recently": 0,
}
resp = rt.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(low_risk),
)
result = json.loads(resp["Body"].read())[0]
print("LOW-RISK PROFILE (daytime, local, grocery with debit card)")
print(f'  Prediction  : {result["label"]}')
print(f'  Probability : {result["probability"]:.1%}')

LOW-RISK PROFILE (daytime, local, grocery with debit card)
  Prediction  : Non-Fraud
  Probability : 1.6%


**Finding — two independent smoke tests via this notebook's own call path both match Notebook 03's direct-boto3 results: a high-risk profile (failed attempts, recent PIN change, Crypto Exchange) scores 95.4% fraud probability, and a low-risk profile (daytime, local grocery, debit card) scores 1.6%.** Corroborating the same result through a second, independently coded client is useful evidence that the deployed model's behaviour is stable and not an artefact of one particular calling code path.

In [9]:
COUNTRIES = ['Australia', 'Brazil', 'Canada', 'France', 'Germany', 'India', 'Japan', 'Mexico', 'UK', 'USA']
CITIES = ['Berlin', 'Delhi', 'Guadalajara', 'London', 'Los Angeles', 'Lyon', 'Manchester', 'Melbourne', 'Mexico City', 'Mumbai', 'Munich', 'New York', 'Osaka', 'Paris', 'Rio', 'Sydney', 'São Paulo', 'Tokyo', 'Toronto', 'Vancouver']
MERCHANTS = ['ATM Withdrawal', 'Clothing', 'Crypto Exchange', 'Education', 'Electronics', 'Entertainment', 'Fuel', 'Gaming', 'Grocery', 'Healthcare', 'Jewelry', 'Online Shopping', 'Restaurant', 'Travel', 'Utilities']
DEVICES = ['ATM', 'Desktop', 'Mobile', 'POS Terminal', 'Tablet']
PAYMENTS = ['Bank Transfer', 'Cheque', 'Credit Card', 'Crypto', 'Debit Card', 'Mobile Payment']

def parse_time(time_str):
    time_str = time_str.strip()
    for fmt in ("%H:%M:%S", "%H:%M"):
        try:
            t = datetime.strptime(time_str, fmt)
            return t.hour, t.strftime("%H:%M:%S"), int(t.hour <= 5)
        except ValueError:
            continue
    raise ValueError(f"Could not parse time: {time_str}")

def predict_fraud(
    transaction_date,
    transaction_time,
    country,
    city,
    merchant_category,
    payment_method,
    device_type,
    customer_age,
    credit_score,
    account_age_years,
    account_balance,
    transaction_amount,
    num_prev_transactions,
    transaction_freq_monthly,
    distance_from_home_km,
    time_since_last_txn_hrs,
    is_international,
    failed_attempts,
    pin_changed_recently,
):
    hour_of_day, transaction_time_fmt, is_night_transaction = parse_time(transaction_time)
    dt = datetime.strptime(transaction_date, "%Y-%m-%d")
    is_weekend = int(dt.weekday() >= 5)

    input_dict = {
        'transaction_id': f"TXN", 
        'customer_id': f"", 
        'transaction_date': transaction_date,
        'transaction_time': transaction_time_fmt,
        'hour_of_day': int(hour_of_day),
        'is_weekend': int(is_weekend),
        'is_night_transaction': int(is_night_transaction),
        'country': country,
        'city': city,
        'merchant_category': merchant_category,
        'payment_method': payment_method,
        'device_type': device_type,
        'customer_age': int(customer_age),
        'credit_score': int(credit_score),
        'account_age_years': float(account_age_years),
        'account_balance': float(account_balance),
        'transaction_amount': float(transaction_amount),
        'num_prev_transactions': int(num_prev_transactions),
        'transaction_freq_monthly': int(transaction_freq_monthly),
        'distance_from_home_km': float(distance_from_home_km),
        'time_since_last_txn_hrs': float(time_since_last_txn_hrs),
        'is_international': int(is_international),
        'failed_attempts': int(failed_attempts),
        'pin_changed_recently': int(pin_changed_recently), 
    }

    try:
        response = rt.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Body=json.dumps(input_dict)
        )
        result = json.loads(response["Body"].read())[0]
        prediction = result.get("label", "Unknown")
        probability = result.get("probability")
        probability_text = f"{probability:.1%}" if probability is not None else "Not returned"

        return (
            f"Endpoint: {ENDPOINT_NAME}\n"
            f"Prediction: {prediction}\n"
            f"Probability: {probability_text}\n"
            f"Input JSON:\n{json.dumps(input_dict, indent=2)}"
        )
    except Exception as e:
        return (
            f"Error invoking endpoint.\n"
            f"Endpoint: {ENDPOINT_NAME}\n"
            f"Error: {str(e)}"
        )

**Insight — the client only collects raw transaction fields (country, city, merchant category, device type, payment method, financial fields, behavioural flags) — never pre-engineered features.** All feature engineering (risk_score, interaction terms, scaling, encoding) happens inside `inference.py` on the endpoint (Notebook 03/05), so this form mirrors exactly what a bank's real transaction system would already have on hand, rather than requiring the client to replicate the training-time feature pipeline.

In [10]:
with gr.Blocks(title="ITI113 Bank Fraud Prediction Demo") as demo:
    gr.Markdown(
        f"""
        # ITI113 Bank Fraud Prediction  Demo

        This simple Gradio UI invokes a SageMaker Serverless Endpoint.

        **Team:** `{TEAM_ID}`  
        **Student/Profile:** `{STUDENT_ID}`  
        **Endpoint:** `{ENDPOINT_NAME}`  
        **Region:** `{REGION}`

        The UI does not train the model. It collects the conventional bank fraud input fields,
        sends them as a JSON request to the deployed SageMaker endpoint, and displays the prediction result.

        The updated endpoint performs preprocessing internally, including `risk_score` creation,
        `amount_to_balance_ratio` creation, numeric scaling, and feature ordering.
        """
    )

    with gr.Row():
        transaction_date = gr.Text(
            label="Transaction Date (YYYY-MM-DD)", value="2024-06-15"
        )
        transaction_time = gr.Text(
            label="Transaction Time (HH:MM or HH:MM:SS)", value="02:30:00"
        )

    with gr.Row():
        country = gr.Dropdown(label="Country", choices=COUNTRIES, value="Brazil")
        city = gr.Dropdown(label="City", choices=CITIES, value="Rio")
        merchant_category = gr.Dropdown(
            label="Merchant Category", choices=MERCHANTS, value="Crypto Exchange"
        )

    with gr.Row():
        device_type = gr.Dropdown(label="Device Type", choices=DEVICES, value="Mobile")
        payment_method = gr.Dropdown(
            label="Payment Method", choices=PAYMENTS, value="Crypto"
        )
        is_international = gr.Dropdown(
            label="International? (0/1)", choices=[0, 1], value=1
        )

    with gr.Row():
        customer_age = gr.Number(label="Customer Age", value=25)
        credit_score = gr.Number(label="Credit Score", value=60)
        account_age_years = gr.Number(label="Account Age (Years)", value=1.0)

    with gr.Row():
        account_balance = gr.Number(label="Account Balance", value=500.0)
        transaction_amount = gr.Number(label="transaction Amount", value=5000.0)
        num_prev_transactions = gr.Number(
            label="Number of Previous Transactions", value=10
        )

    with gr.Row():
        transaction_freq_monthly = gr.Number(
            label="Transaction Frequency (monthly)", value=2
        )
        distance_from_home_km = gr.Number(label="Distance from Home (km)", value=100.0)
        time_since_last_txn_hrs = gr.Number(
            label="Time since Last Transaction (hrs)", value=24.0
        )

    with gr.Row():
        failed_attempts = gr.Number(label="Failed Attempts", value=3)
        pin_changed_recently = gr.Number(label="Pin Changed Recently? (0/1)", value=1)

    predict_button = gr.Button("Predict")
    output = gr.Textbox(label="Prediction Result", lines=12)

    predict_button.click(
        fn=predict_fraud,
        inputs=[
            transaction_date,
            transaction_time,
            country,
            city,
            merchant_category,
            payment_method,
            device_type,
            customer_age,
            credit_score,
            account_age_years,
            account_balance,
            transaction_amount,
            num_prev_transactions,
            transaction_freq_monthly,
            distance_from_home_km,
            time_since_last_txn_hrs,
            is_international,
            failed_attempts,
            pin_changed_recently,
        ],
        outputs=output,
    )
# In SageMaker Studio, inline=True is usually the most convenient for notebook demos.

# if below fails to show:
# demo.launch(
#     inline=True,
#     share=False,
#     debug=True,
#     server_name="0.0.0.0",
#     server_port=7860
# )
# run this temporarily
demo.launch(
    inline=True,
    share=True,
    debug=True,
    server_name="0.0.0.0",
    server_port=7860,
)


* Running on local URL:  http://0.0.0.0:7860


* Running on public URL: https://f3d1ff69189ea8a27a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Insight — the public `gradio.live` share link is treated as a short-lived demo artefact, not a standing deployment.** Because the link is temporary (up to ~1 week) and unauthenticated, the notebook explicitly calls `demo.close()` and force-releases the local port immediately after each demo session, which is a deliberate, documented mitigation for the access-control risk an always-on public link would otherwise introduce.

In [ ]:
demo.close()
print("Gradio demo closed.")

In [ ]:
# run this to confirm gradio closed
import os
import signal
import subprocess

PORT = 7860

cmd = f"lsof -ti:{PORT}"
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

pids = result.stdout.strip().splitlines()

if not pids:
    print(f"No process found on port {PORT}")
else:
    for pid in pids:
        print(f"Killing process {pid} on port {PORT}")
        os.kill(int(pid), signal.SIGTERM)

    print(f"Port {PORT} should now be free.")